# 02 – Feature Engineering

In this notebook we:

- Apply the preprocessing pipeline from `src/preprocess.py`.
- Add handcrafted features in `src/feature_engineering.py`.
- Explore the new features.
- Explain why these features can help the model.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import load_raw_data, merge_datasets, basic_cleaning, split_train_valid
from src.feature_engineering import (
    add_handcrafted_features,
    get_feature_lists,
)

plt.rcParams["figure.figsize"] = (10, 6)
sns.set(style="whitegrid")

In [ ]:
train_transaction, train_identity, test_transaction, test_identity = load_raw_data(
    data_dir="../data/raw"
)
train_merged = merge_datasets(train_transaction, train_identity)
train_clean = basic_cleaning(train_merged)
train_df, valid_df = split_train_valid(train_clean, target_col="isFraud")

train_df.head()

In [ ]:
train_fe = add_handcrafted_features(train_df.copy())
valid_fe = add_handcrafted_features(valid_df.copy())

train_fe[["TransactionAmt", "TransactionAmt_log", "Transaction_day", "Transaction_hour"]].head()

In [ ]:
# Compare original and log-transformed transaction amount
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train_fe["TransactionAmt"].dropna(), bins=100, ax=axes[0])
axes[0].set_title("Raw TransactionAmt")

sns.histplot(train_fe["TransactionAmt_log"].dropna(), bins=100, ax=axes[1])
axes[1].set_title("Log-transformed TransactionAmt")

plt.tight_layout()
plt.show()

In [ ]:
# Average fraud rate by day and hour
day_fraud_rate = train_fe.groupby("Transaction_day")["isFraud"].mean()
hour_fraud_rate = train_fe.groupby("Transaction_hour")["isFraud"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_fraud_rate.plot(ax=axes[0])
axes[0].set_title("Average fraud rate by day")
axes[0].set_xlabel("Day")

hour_fraud_rate.plot(ax=axes[1])
axes[1].set_title("Average fraud rate by hour")
axes[1].set_xlabel("Hour")

plt.tight_layout()
plt.show()

## Why these features help

- `TransactionAmt_log`: The log transformation reduces the effect of very large amounts.
  This often makes patterns easier for the model to learn.
- `Transaction_day` and `Transaction_hour`: Fraud can be time-dependent.
  For example, some days or hours of the day may have higher fraud risk due to specific attacks.

We will use these features in the training pipeline.
